# Batch Molecular Generation

Load core and substituent definitions from `data/`, generate substituted products, and save them in `output/`. See the repository README before running.

In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import chain, combinations
import random
import re
import time

import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize

RDLogger.DisableLog('rdApp.*')

# Run this notebook from the project root; do not use machine-specific absolute paths.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'data').is_dir():
    raise FileNotFoundError('Start Jupyter from the molecular-generation directory, or adjust PROJECT_DIR.')

CORE_FILE = PROJECT_DIR / 'data' / 'cores.xlsx'  # Core table; the first two columns are read.
SUBSTITUENT_FILE = PROJECT_DIR / 'data' / 'substituents.xlsx'  # Substituent table; the first two columns are read.
OUTPUT_FILE = PROJECT_DIR / 'output' / 'generated_molecules.csv'  # CSV destination; existing output is replaced.
CORE_SMILES_COLUMN = 'smiles'  # Informational only: core SMILES are read from the first column.
CORE_SITES_COLUMN = 'sub_idx'  # Informational only: substitution sites are read from the second column.
CORE_INDICES = None  # Core row indices to process; use None to process every core.
MOLECULES_PER_CORE = 10  # Target number of unique products for each core.
RARE_GROUP_IDS = []  # Optional 1-based substituent row numbers sampled as rare groups.
RARE_GROUP_RATIO = 0.0  # Probability of sampling a rare group at an eligible site.
NONSYMMETRIC_RATIO = 0.2  # Probability of assigning distinct groups to paired symmetric sites.
RANDOM_SEED = 20250729  # Set to None to disable reproducible random sampling.
MAX_ATTEMPTS_PER_CORE = 10_000  # Stop after this many attempted products for one core.
TIMEOUT_SECONDS_PER_CORE = 240  # Stop after this many seconds for one core.

rng = random.Random(RANDOM_SEED)

In [ ]:
def parse_sites(value):
    """Parse substitution sites such as `1, 3, (5,7)`."""
    if pd.isna(value):
        return []
    matches = re.findall(r'\((\d+(?:,\d+)*)\)|(\d+)', str(value))
    return [tuple(map(int, pair.split(','))) if pair else int(single) for pair, single in matches]

def is_blank_site_value(value):
    """Return True only when the second core-table cell is empty."""
    return pd.isna(value) or not str(value).strip()

def infer_substitution_sites(mol):
    """Infer aromatic-carbon sites and group symmetry-equivalent positions."""
    candidates = sorted(atom.GetIdx() for atom in mol.GetAtoms()
                        if atom.GetAtomicNum() == 6 and atom.GetIsAromatic() and atom.GetDegree() == 2)
    ranks, groups = Chem.CanonicalRankAtoms(mol, breakTies=False), {}
    for atom_index in candidates:
        groups.setdefault(ranks[atom_index], []).append(atom_index)
    return [tuple(indices) if len(indices) > 1 else indices[0]
            for indices in sorted(groups.values(), key=lambda indices: indices[0])]

def format_sites(sites):
    return ', '.join(f"({','.join(map(str, site))})" if isinstance(site, tuple) else str(site) for site in sites)

def load_add_functions(path):
    table = pd.read_excel(path)
    if table.shape[1] < 2:
        raise ValueError('substituents.xlsx must contain at least two columns.')
    functions = {}
    for row_number, row in table.iterrows():
        smiles, attachment_index = row.iloc[0], int(row.iloc[1])
        fragment = Chem.MolFromSmiles(str(smiles))
        if fragment is None or not 0 <= attachment_index < fragment.GetNumAtoms():
            raise ValueError(f'Invalid substituent in row {row_number + 1}.')
        def add_group(mol, atom_index, fragment=fragment, attachment_index=attachment_index):
            editable = Chem.RWMol(mol)
            offset = editable.GetNumAtoms()
            editable.InsertMol(fragment)
            editable.AddBond(atom_index, offset + attachment_index, Chem.BondType.SINGLE)
            Chem.SanitizeMol(editable)
            return Chem.RemoveHs(editable)
        functions[row_number + 1] = add_group
    return functions

class SmilesCleaner:
    def __init__(self):
        self.normalizer = MolStandardize.rdMolStandardize.Normalizer()
        self.largest_fragment = MolStandardize.rdMolStandardize.LargestFragmentChooser()
        self.uncharger = MolStandardize.rdMolStandardize.Uncharger()

    def clean(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = self.normalizer.normalize(mol)
        mol = self.largest_fragment.choose(mol)
        mol = self.uncharger.uncharge(mol)
        return Chem.MolToSmiles(mol, isomericSmiles=False, canonical=True)

def candidate_site_sets(sites, rng):
    pool = rng.sample(sites, 10) if len(sites) > 10 else sites
    result = list(chain.from_iterable(combinations(pool, count) for count in range(1, len(pool) + 1)))
    rng.shuffle(result)
    return result

def choose_groups(site_set, group_ids, rare_ids, rare_ratio, nonsymmetric_ratio, rng):
    chosen, kinds = {}, set()
    for site in site_set:
        targets = site if isinstance(site, tuple) and rng.random() < nonsymmetric_ratio else [site]
        for target in targets:
            proposed = rng.choice(rare_ids) if rare_ids and rng.random() < rare_ratio else rng.choice(group_ids)
            group = proposed if len(kinds) < 3 or proposed in kinds else rng.choice(tuple(kinds))
            if isinstance(site, tuple) and len(targets) == 1:
                chosen.update({paired_site: group for paired_site in site})
                kinds.add(group)
                break
            chosen[target] = group
            kinds.add(group)
    return chosen


In [ ]:
def process_molecule(core_index, cores, add_functions, group_ids, *, num, rng):
    row = cores.iloc[core_index]
    mol = Chem.MolFromSmiles(str(row.iloc[0]))
    if mol is None:
        raise ValueError(f'Invalid SMILES for core {core_index}.')
    site_value = row.iloc[1]
    if is_blank_site_value(site_value):
        sites = infer_substitution_sites(mol)
        if not sites:
            raise ValueError(f'No substitutable aromatic-carbon sites found for core {core_index}.')
        print(f'Core {core_index}: inferred substitution sites: {format_sites(sites)}')
    else:
        sites = parse_sites(site_value)
        if not sites:
            raise ValueError(f'Invalid substitution-site format for core {core_index}.')
    missing = set(group_ids + RARE_GROUP_IDS) - set(add_functions)
    if missing:
        raise ValueError(f'Substituent table is missing IDs: {sorted(missing)}')

    candidates = candidate_site_sets(sites, rng)
    generated, started, attempts = set(), time.monotonic(), 0
    while len(generated) < num and attempts < MAX_ATTEMPTS_PER_CORE and time.monotonic() - started < TIMEOUT_SECONDS_PER_CORE:
        attempts += 1
        try:
            assignments = choose_groups(rng.choice(candidates), group_ids, RARE_GROUP_IDS, RARE_GROUP_RATIO, NONSYMMETRIC_RATIO, rng)
            product = Chem.RWMol(mol)
            for atom_index, group_id in assignments.items():
                product = add_functions[group_id](product, atom_index)
            cleaned = SmilesCleaner().clean(Chem.MolToSmiles(product))
            if cleaned:
                generated.add(cleaned)
        except Exception:
            continue
    if len(generated) < num:
        print(f'Core {core_index}: generated only {len(generated)}/{num} unique molecules after {attempts} attempts.')
    return pd.DataFrame({'name': [f'core{core_index + 1}_{j + 1}' for j in range(len(generated))], 'smiles': list(generated)})

if not CORE_FILE.exists() or not SUBSTITUENT_FILE.exists():
    raise FileNotFoundError('Input Excel files are missing. Place them in data/ as described in data/README.md, or update the configuration cell.')
cores = pd.read_excel(CORE_FILE)
if cores.shape[1] < 2:
    raise ValueError('cores.xlsx must contain at least two columns.')
add_functions = load_add_functions(SUBSTITUENT_FILE)
group_ids = sorted(add_functions)  # Every substituent row is available automatically.
core_indices = range(len(cores)) if CORE_INDICES is None else CORE_INDICES

results = []
for core_index in core_indices:
    print(f"{datetime.now():%H:%M:%S}: Processing core {core_index}")
    try:
        results.append(process_molecule(core_index, cores, add_functions, group_ids, num=MOLECULES_PER_CORE, rng=rng))
    except (IndexError, ValueError) as error:
        print(f'Failed to process core {core_index}: {error}')

result = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=['name', 'smiles'])
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUTPUT_FILE, index=False)
print(f'Exported {len(result)} molecules to: {OUTPUT_FILE}')
result.head()